# PCA, Clustering and Embeddings

Today we'll analyse the BBC News Articles dataset.

We will turn each article into an embedding, a numeric location on a high-dimensional meaning map, reduce those embeddings with PCA, and use them for clustering and classification.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import urllib

np.random.seed(42)

pd.set_option("display.max_colwidth", 200)

from sentence_transformers import SentenceTransformer


## The data

[BBC News Articles](https://huggingface.co/datasets/SetFit/bbc-news)

In [2]:
df = pd.read_csv("https://huggingface.co/datasets/SetFit/bbc-news/resolve/main/bbc-text.csv")

In [3]:
df.sample(10)

,category,text
414,politics,brown and blair face new rift claims for the umpteenth time tony blair and gordon brown are said to have declared all out war on each other. this time the alleged rift is over who should take th...
420,business,small firms hit by rising costs rising fuel and materials costs are hitting confidence among the uk s small manufacturers despite a rise in output business lobby group the cbi says. a cbi quar...
1644,entertainment,spirit awards hail sideways the comedy sideways has dominated this year s independent spirit awards winning all six of the awards for which it was nominated. it was named best film while alexand...
416,tech,microsoft releases patches microsoft has warned pc users to update their systems with the latest security fixes for flaws in windows programs. in its monthly security bulletin it flagged up eigh...
1232,sport,arsenal through on penalties arsenal win 4-2 on penalties the spanish goalkeeper saved from alan quinn and jon harley as arsenal sealed a quarter-final trip to bolton with a 4-2 victory on penalt...
1544,business,jobs go at oracle after takeover oracle has announced it is cutting about 5 000 jobs following the completion of its $10.3bn takeover of its smaller rival peoplesoft last week. the company said i...
1748,business,id theft surge hits us consumers almost a quarter of a million us consumers complained of being targeted for identity theft in 2004 official figures suggest. the federal trade commission said tw...
1264,sport,poll explains free-kick decision referee graham poll said he applied the laws of the game in allowing arsenal striker thierry henry s free-kick in sunday s 2-2 draw with chelsea. keeper petr cech...
629,sport,parmar ruled out of davis cup tie a knee injury has forced arvind parmar out of great britain s davis cup tie in israel and left alex bogdanovic in line to take the second singles place. parmar p...
1043,tech,video phones act as dating tools technologies from e-mail to net chatrooms instant messaging and mobiles have proved to be a big pull with those looking for love. the lure once was that you c...


In [4]:
df.value_counts("category")

category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

In [5]:
df.sample()["text"].values[0]

'mourinho receives robson warning sir bobby robson has offered chelsea boss jose mourinho some advice on coping under pressure.  the pair worked together at barcelona and porto and robson had a word of warning for his protege.  it has all gone for him just lately and that is marvellous  but sometimes you have to have a bit of humility and learn how to lose   said robson.  it is when it goes against you and you get a bit of bad luck that you learn  and he ll get it straight.  robson was speaking after being formally granted the freedom of the city of newcastle.  jose is doing very well at the moment   robson added of the man who worked for him for six years.  he has got one pot - possibly two to follow - a big game against barcelona to come and i cannot see them losing their lead in the premiership.  they are in a good position and i would expect them to go on and win it  which is a wonderful achievement.   what has occurred over the last couple of weeks will stand him in very good stea

## Embeddings

An embedding is a numeric representation of a piece of text. You can also think of it as a representation of that text in a multidimensional space. Articles with similar meaning should end up close together, and very different articles should end up far apart.


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

emb_df = pd.DataFrame(embeddings)


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

In [8]:
# If above cell takes too long to run, use the precomputed embeddings
url = 'https://pub-c88b3a7f2ed141418355b2bfb03c96e6.r2.dev/embeddings.csv'
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response:
    emb_df = pd.read_csv(response, index_col=0)


In [9]:
df.iloc[0]["text"]

'tv future in the hands of viewers with home theatre systems  plasma high-definition tvs  and digital video recorders moving into the living room  the way people watch tv will be radically different in five years  time.  that is according to an expert panel which gathered at the annual consumer electronics show in las vegas to discuss how these new technologies will impact one of our favourite pastimes. with the us leading the trend  programmes and other content will be delivered to viewers via home networks  through cable  satellite  telecoms companies  and broadband service providers to front rooms and portable devices.  one of the most talked-about technologies of ces has been digital and personal video recorders (dvr and pvr). these set-top boxes  like the us s tivo and the uk s sky+ system  allow people to record  store  play  pause and forward wind tv programmes when they want.  essentially  the technology allows for much more personalised tv. they are also being built-in to high

In [10]:
df.iloc[0]["category"]

'tech'

In [11]:
emb_df

,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.001554,-0.067275,0.011174,-0.097146,0.054098,0.042254,-0.034863,-0.017126,0.062448,-0.024317,...,0.083276,-0.012365,0.099596,-0.002854,0.030708,0.069352,-0.008165,-0.015212,-0.063724,0.084557
1,-0.083547,0.059484,-0.013196,-0.011908,0.011631,0.002616,0.117315,0.002310,0.013166,0.028018,...,0.056850,-0.037205,0.039960,0.014939,-0.075912,-0.010551,0.011098,-0.093159,-0.002457,0.021243
2,-0.055962,-0.008481,-0.025843,-0.056737,-0.045265,-0.021684,0.030081,-0.013983,0.030562,-0.003697,...,0.014019,-0.006897,-0.006589,-0.018533,-0.056650,-0.000594,0.039833,-0.056027,0.043895,-0.046996
3,0.015004,-0.125696,-0.028034,-0.040649,0.080588,0.052048,0.025743,-0.012287,0.033973,0.002043,...,0.004305,-0.004015,-0.057729,-0.021071,0.009793,0.056244,-0.032304,-0.020664,-0.025317,0.056185
4,-0.018273,-0.018895,-0.047967,-0.070612,-0.006405,0.059953,-0.119612,0.026853,0.050197,-0.011960,...,0.051296,0.027649,-0.022006,-0.028721,-0.003901,0.082542,-0.065968,-0.073110,-0.035203,0.043847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2220,-0.027352,-0.026011,0.054015,0.025434,-0.014118,0.073263,-0.022070,0.068582,-0.033815,-0.002269,...,-0.010152,-0.027655,-0.055435,-0.022257,-0.029578,-0.062166,-0.029384,-0.129902,-0.041675,0.080504
2221,-0.017127,0.009932,0.017573,-0.024224,0.050684,0.034838,0.082775,-0.099431,-0.101437,0.020063,...,0.045735,-0.004506,-0.019301,-0.011101,-0.026871,0.096571,-0.026657,0.012036,-0.049906,0.012583
2222,0.027212,-0.136341,0.026965,-0.063657,-0.002635,0.035392,0.036003,-0.031810,0.021682,0.062556,...,0.032426,0.031709,-0.021834,0.052545,0.000720,0.055665,-0.050568,-0.092031,-0.077371,-0.007813
2223,0.046371,-0.035961,0.065271,-0.032036,0.030133,0.061473,-0.000475,-0.001458,0.036582,0.081157,...,0.069588,0.028942,0.002779,-0.033584,-0.061675,0.092261,0.012804,0.045767,-0.016248,0.048731


In [12]:
emb_df.iloc[0].tolist()

[-0.001553876,
 -0.06727469,
 0.011174214,
 -0.09714647,
 0.05409849,
 0.042253822,
 -0.03486287,
 -0.017126063,
 0.062448256,
 -0.024316564,
 -0.04787935,
 0.045247436,
 -0.01705424,
 -0.015675152,
 0.013085875,
 -0.1557984,
 0.07755032,
 -0.12457919,
 -0.007810337,
 0.023632307,
 -0.013241261,
 -0.055903994,
 -0.052496992,
 -0.004659107,
 0.05348777,
 -0.022686033,
 -0.06298673,
 0.0005361094,
 -0.0037238155,
 -0.030825287,
 -0.0024546355,
 0.03775705,
 0.013111599,
 0.044811916,
 -0.07275285,
 -0.039356403,
 0.00037462913,
 -0.004450885,
 -0.10191604,
 0.0017646332,
 0.06019026,
 -0.09947903,
 -0.009305393,
 -0.061449282,
 -0.0027737643,
 0.040667787,
 0.021240814,
 -0.040535644,
 -0.07212074,
 -0.0015725094,
 -0.0652869,
 0.030321548,
 0.06136828,
 0.020016698,
 -0.021162895,
 -0.0056913635,
 0.047824703,
 0.08727437,
 -0.0030796633,
 0.029061183,
 -0.0060282955,
 -0.041554086,
 -0.028440239,
 0.036712542,
 -0.015836779,
 0.03216546,
 0.066254795,
 0.11543839,
 0.021183234,
 -0.072

## Tasks


### Task 1

Use distances between embeddings to find articles that are most similar to random article from a dataset.

For distances, you can use [sklearn.metrics.pairwise_distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html). Are the articles similar?


In [13]:
from sklearn.metrics import pairwise_distances

article_index = np.random.choice(range(0, len(df)))

### Task 2

PCA reduces many embedding features down to two components so we can visualize them.

- Reduce the dimensions of the embeddings dataset to two principal components.
- Assign the principal component values to the original dataframe `df`.
- Plot the result on a scatterplot.


### Task 3

Print out several articles that are close to one another on the scatterplot above. Is their content similar?


### Task 4

Apply k-means clustering.

- Cluster the articles into three clusters using k-means clustering.
- Visualize the clusters on a scatterplot.
- Do the clusters have something in common in terms of article category?


### Task 5

Find the optimal number of clusters via the Elbow method.

### Task 6

Fit kmeans with another value for `n_clusters` and visualize the clusters. Show which article categories belong to the clusters and interpret.




### Task 7

Repeat the PCA + clustering workflow in three dimensions.

- Reduce the embedding dataset to three principal components.
- Find optimal number of clusters.
- Fit k-means with that number of clusters.
- Visualize the result with a Plotly 3D scatterplot.


In [ ]:
import plotly.express as px

optimal_k_3d = 4

kmeans_3 = KMeans(n_clusters=optimal_k_3d)
df["cluster"] = kmeans_3.fit_predict(df[["pc1", "pc2", "pc3"]]).astype('str')

fig = px.scatter_3d(
    df,
    x="pc1",
    y="pc2",
    z="pc3",
    color="cluster",
)

fig.update_traces(marker={"size": 3})
fig.update_layout(width=1000, height=800)

fig.show()


### Task 8

Now we use the embedding coordinates as input features for a supervised model.

Train a model (RandomForest or LogisticRegression) for predicting whether an article belongs to the politics category. The model `pipeline` should take the full embedding as input and perform PCA before feeding it into the next steps of the pipeline. You should also perform hyperparameter tuning for n_components in PCA and one other hyperparameter.

Report the performance of the model on the test set:
- Report the performance metrics;
- Do some error analysis - what do you notice about the false positives and false negatives in terms of content and their categories?

Hint: you can use PCA in a scikit-learn pipeline just like you would e.g. `PolynomialFeatures`.

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures

y = df["category"] == "politics"
X = emb_df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


## Rest of the Course

- Two intensive weeks in May.
- Will upload the last homework today or tomorrow.
- Will share information about group project consultations next week.
